# Match Overview — Phase 1 demo

Runs the full Phase 1 analytics pipeline on an ingested match and renders the key visualizations.

Prerequisite: `fa-data fetch` + `fa-data ingest` for the match of interest.

Example:
```bash
uv run fa-data fetch statsbomb --competition-id 43 --season-id 106 --match-id 3869685 --force
uv run fa-data ingest statsbomb --match-id 3869685
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from football_analysis.analytics.pipeline.runner import run as run_pipeline
from football_analysis.app.data import list_ingested_matches, load_match_events, team_names_for_match
from football_analysis.viz.static.shot_map import plot_shot_map
from football_analysis.viz.static.pass_network import plot_pass_network
from football_analysis.viz.static.heatmap import plot_player_heatmap
from football_analysis.viz.static.xt_surface import plot_xt_surface

## 1. Pick a match

In [ ]:
matches = list_ingested_matches()
matches

In [ ]:
# Change to any `match_id` from the table above
match_id = 'statsbomb:3869685'  # Argentina vs France, WC2022 final
events = load_match_events(match_id)
names = team_names_for_match(match_id)
print(f'{len(events)} events, teams: {names}')

## 2. Run the pipeline

In [ ]:
analytics = run_pipeline(events)
enriched = analytics.events

print('PPDA        :', {names.get(t, t): round(v, 2) for t, v in analytics.ppda.items()})
print('Field tilt  :', {names.get(t, t): f'{v:.1%}' for t, v in analytics.field_tilt.items()})

## 3. Shot map

In [ ]:
fig = plot_shot_map(enriched)
plt.show()

## 4. Pass networks

In [ ]:
for tid in sorted(enriched['team_id'].dropna().unique()):
    fig = plot_pass_network(enriched, team_id=tid, min_passes_edge=5, title=f'Pass network — {names.get(tid, tid)}')
    plt.show()

## 5. xT surface (learned from this match)

In [ ]:
fig = plot_xt_surface(analytics.xt_grid)
plt.show()

## 6. Top xT contributors

In [ ]:
pos_moves = enriched[enriched['xt_delta'].notna() & (enriched['xt_delta'] > 0)]
ranked = (
    pos_moves.groupby('player_id')['xt_delta']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'xt_total', 'count': 'actions'})
    .sort_values('xt_total', ascending=False)
    .head(10)
)
ranked

In [ ]:
top_player = str(ranked.index[0])
fig = plot_player_heatmap(enriched, player_id=top_player, title=f'Heatmap — player {top_player}')
plt.show()